In [ ]:
%pip install transformers

In [ ]:
import pandas as pd
from transformers import T5Tokenizer,Trainer,TrainingArguments,T5ForConditionalGeneration

In [ ]:
train_data=pd.read_csv("samsum-train.csv")
val_data=pd.read_csv("samsum-validation.csv")

In [ ]:
train_data.head()

In [ ]:
train_data["dialogue"][0]

In [ ]:
train_data.shape

In [ ]:
val_data.shape

In [ ]:
#random sampling

train_data=train_data.sample(n=4000,random_state=42).reset_index(drop=True)
val_data=val_data.sample(n=500,random_state=42).reset_index(drop=True)

In [ ]:
train_data.shape

In [ ]:
val_data.shape

# data pre-processing

In [ ]:
import re

def clean_data(text):
    text=re.sub(r"\r\n"," ",text) #lines
    text=re.sub(r"\s+"," ",text) #spaces
    text=re.sub(r"<.*?>"," ",text) #html tags <p> <h1>
    text=text.strip().lower()
    return text

In [ ]:
train_data["dialogue"]=train_data["dialogue"].apply(clean_data)
train_data["summary"]=train_data["summary"].apply(clean_data)

val_data["dialogue"]=val_data["dialogue"].apply(clean_data)
val_data["summary"]=val_data["summary"].apply(clean_data)

In [ ]:
train_data["dialogue"][0]

# tokenize

In [ ]:
tokenizer=T5Tokenizer.from_pretrained("t5-small")

In [ ]:
# raw data -> tokenized inputs for fine-tuning

def tokenize(data):
    inputs=tokenizer(data["dialogue"],padding="max_length",max_length=512,truncation=True)
    targets=tokenizer(data["summary"],padding="max_length",max_length=150,truncation=True)

    inputs["labels"]=targets["input_ids"] #token ids=> add to input as labels
    return inputs

In [ ]:
train_dataset=train_data.apply(tokenize,axis=1).tolist()
val_dataset=val_data.apply(tokenize,axis=1).tolist()

In [ ]:
train_dataset[0]

In [ ]:
#input ids - dialogue =>token ids

#1=> EOS end of sequence,0-> padding

#attnetion mask -> if the attention mask is 1 the toke id is valid and if the token id is invalid the mask value is 0
#labels-target=>summary


In [ ]:
len(val_dataset[0]["labels"])

In [ ]:
type(train_dataset)

In [ ]:
type(val_dataset)

# working with our model

In [ ]:
# we are performing text summa... task which is an NLP task => generation task

model=T5ForConditionalGeneration.from_pretrained("t5-small")

In [ ]:
import torch

if torch.backends.mps.is_available():
    device=torch.device("mps")
elif torch.cuda.is_available():
    device=torch.device("cuda")
else:
    device=torch.device("cpu")

print("device: ",device)
model.to(device)

In [ ]:
!pip install "accelerate>=1.1.0"


In [ ]:
import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "accelerate>=1.1.0"
])

In [ ]:
import accelerate
print(accelerate.__version__)

In [ ]:
# training arguments

training_args=TrainingArguments(
    output_dir="./results",

    num_train_epochs=6,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500
)

In [ ]:
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)